# Proyecto — Data Stream Processor

## Contexto (extremadamente importante): 

Uno de las mayores virtudes de un repositorio en github es la poder volver sobre los cambios hechos. Uno puede volver sbre algún `commit` he iniciar el proceso desde ese punto. 

En otro ejemplo parecido, al crear una lista en python, dicha lista se modifica al agregar o borrar elementos y regresar a un estado anterior de la lista no es tan sencillo. La idea de este proyecto es poder emular dicho proceso y realizar una especie de lista con memoria para poder llevar algunos registros de manera adecuada.


### Objetivo

Construya un pequeño sistema para recibir y procesar registros de datos utilizando las clases `ArrayStack` y `ArrayQueue` proporcionadas por el curso. La idea central es poder llevar la información en orden para llevar los cambios de los registros de manera adecuada.

En el método `__init__` debe aparecer tres atributos:
- **Queue:** registros que han llegado pero todavía no han sido procesados.
- **Stack:** historial de cambios realizados, para poder deshacer los cambios más recientes.
- **Lista:** como se encuentran los registros actualmente

Las clases `ArrayStack` y `ArrayQueue` ya están implementadas. **No debe implementarlas nuevamente ni modificarlas.**

## 1. Registro de datos

Cada registro es una tupla de tres elementos:

```python
(sensor, variable, value)
```

Ejemplos:

```python
("S01", "temperature", 23.5)
("S02", "temperature", 25.1)
("S01", "humidity", 61.2)
```

Una combinación única `(sensor, variable)` identifica un dato dentro del estado actual.

## 2. Clase `DataProcessor`

Implemente:

```python
class DataProcessor:
    ...
```

Debe utilizar:

- un `ArrayQueue` para los registros pendientes;
- un `ArrayStack` para el historial de cambios;
- un `list` para mantener el estado actual.

### Restricción

No sustituya `ArrayQueue` o `ArrayStack` por `list`, `collections.deque` u otra estructura para realizar las funciones que corresponden a la Queue o al Stack.

La clase `DataProcessor` no debe imprimir resultados. Los métodos deben devolver los valores especificados. Las impresiones utilizadas para demostrar el funcionamiento deben realizarse en las celdas de prueba.

## 3. `add(record)`

Agrega un registro a la `ArrayQueue`.

```python
processor.add(("S01", "temperature", 23.5))
```

Requisitos:

- agrega el registro a la cola;
- **no procesa** el registro;
- conserva el orden de llegada.

No se requiere un valor de retorno.

Debe rechazar registros que no tengan exactamente tres componentes o cuyo `value` no sea numérico. El tipo concreto de excepción para estos errores puede ser elegido por el estudiante, pero debe documentarse y utilizarse consistentemente.

## 4. `process_next()`

Procesa el siguiente registro pendiente.

Debe:

1. obtener el siguiente registro de la Queue;
2. procesarlo;
3. actualizar el estado actual;
4. guardar en el Stack la información necesaria para poder deshacer exactamente ese cambio.

### FIFO

Si se ejecuta:

```python
add(A)
add(B)
add(C)
```

las llamadas sucesivas a `process_next()` deben devolver/procesar `A`, luego `B` y luego `C`.

### Actualización

Si se procesa:

```python
("S01", "temperature", 23.5)
```

en el estado se debe reflejar:

```python
('S01', 'temperature', 23.5)
```

Si después se procesa `("S01", "temperature", 27.0)`, el valor actual debe ser `27.0`.

### Historial

Se debe agregar al `Stack` respectivo

### Retorno

Debe devolver el registro que acaba de ser procesado.

### Queue vacía

Si no hay registros pendientes, debe producir `Empty, el error creado en el repositorio `Goodrich`

## 5. `undo()`

Deshace el último cambio realizado mediante `process_next()`.

Ejemplo:

```text
20 → 25 → 30
```

Después de un `undo()`:

```text
20 → 25
```

Después de otro:

```text
20
```

Los cambios deben deshacerse en orden LIFO.

### Dato creado por primera vez

Suponga que inicialmente no existe `('S01', 'temperature', x)`, después de procesar:

```python
("S01", "temperature", 23.5)
```

el dato existe. Si se ejecuta `undo()`, debe volver a **no existir**.

### Historial vacío

Si no hay cambios que deshacer, debe producir `Empty`.

No se requiere un valor de retorno.

## 6. `pending()`

Devuelve el número de registros que todavía esperan ser procesados.

Por ejemplo, después de:

```python
add(A)
add(B)
add(C)
```

`pending()` debe devolver `3`. Después de `process_next()`, debe devolver `2`.

## 7. `current_value(sensor, variable)`

Devuelve el valor actual asociado con una combinación de sensor y variable.

Ejemplo:

```python
current_value("S01", "temperature")
```

puede devolver `23.5`.

Si nunca se ha procesado un registro para esa combinación, debe producir `KeyError`.

# 8. Ejemplo completo

Considere:

```python
A = ("S01", "temperature", 20)
B = ("S01", "temperature", 25)
C = ("S01", "humidity", 60)
```

Después de `add(A)`, `add(B)`, `add(C)`, la Queue contiene `A → B → C`.

Después de procesar A, el estado contiene:

```text
S01 / temperature → 20
```

Después de procesar B:

```text
S01 / temperature → 25
```

Después de procesar C:

```text
S01 / temperature → 25
S01 / humidity    → 60
```

Un `undo()` elimina el efecto de C. Otro `undo()` elimina el efecto de B. Otro `undo()` elimina el efecto de A y el estado vuelve a estar vacío.

# 9. Pruebas obligatorias

Incluya pruebas para, como mínimo:

1. `pending()` sobre un procesador vacío.
2. Agregar un registro.
3. Agregar varios registros.
4. Verificar procesamiento FIFO.
5. Procesar un registro.
6. Procesar varios registros.
7. Actualizar una variable existente.
8. Consultar el valor actual.
9. Realizar un `undo()`.
10. Realizar varios `undo()` consecutivos.
11. Procesar cuando la Queue está vacía.
12. Hacer `undo()` cuando el historial está vacío.
13. Deshacer la creación de un dato que antes no existía.
14. Hacer varios cambios sobre la misma variable.
15. Agregar un registro con formato incorrecto.
16. Agregar un registro cuyo valor no sea numérico.
17. Consultar un sensor/variable que nunca haya sido procesado.

# 10. Análisis de complejidad

Explique la complejidad temporal de:

- `add`
- `process_next`
- `undo`
- `pending`
- `current_value`

Justifique qué operaciones determinan cada complejidad e indique qué estructuras auxiliares utiliza el sistema.

# 11. Restricciones

1. Utilice las clases `ArrayStack` y `ArrayQueue` proporcionadas.
2. No las reemplace por `list`, `deque` u otra estructura equivalente.
3. No modifique las implementaciones proporcionadas.
4. La implementación debe estar contenida en `DataProcessor`.
5. Incluya las pruebas solicitadas.
6. Explique brevemente sus decisiones de diseño.

# 12. Bonus — `redo()`

Como extensión opcional, implemente:

```python
redo()
```

Después de un `undo()`, el sistema debe poder volver a aplicar el cambio que acaba de deshacerse.

Por ejemplo:

```text
20 → 25 → 30
undo()  → 20 → 25
redo()  → 20 → 25 → 30
```

El estudiante debe explicar qué estructuras utiliza para implementar `redo()` y por qué. No se proporciona la estrategia de implementación.


# 13. Entrega

La entrega debe contener:

- implementación completa de `DataProcessor`;
- pruebas solicitadas;
- explicación breve de las decisiones de diseño;
- análisis de complejidad temporal;
- si realiza el bonus, implementación y explicación de `redo()`.